# Percepción de seguridad, convivencia y satisfacción de servicios — Centro Histórico de Quito 2024-2026 (HTML + informe técnico)

**Reproducibilidad del producto HTML.** Este cuaderno recalcula, desde la base de datos original, todas las cifras que aparecen en el HTML (texto, tablas, barras, KPI).

**Cómo usarlo:** *Entorno de ejecución → Ejecutar todo*. Cuando lo pida, sube: `261030_CHQ_unificado_LC_CA_PF_limpia.xlsx`, `Unificado_Basilica_2026.xlsx` (y, para la verificación final, `261030_CHQ_percepcion_convivencia_servicios_v4.html`).

Resultado: `salida/app_data.json` (los agregados que incrusta el HTML) y `salida/cifras_html.csv` (una fila por cifra).

In [ ]:
# Instalación y carga de datos
import sys, os, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "statsmodels", "openpyxl"], check=False)

DATA_FILES = ["261030_CHQ_unificado_LC_CA_PF_limpia.xlsx", "Unificado_Basilica_2026.xlsx"]
OUT = "salida"; os.makedirs(OUT, exist_ok=True)

try:
    from google.colab import files as _colab_files   # solo existe en Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

faltan = [f for f in DATA_FILES if not os.path.exists(f)]
if faltan and IN_COLAB:
    print("Sube estos archivos:", faltan)
    subidos = _colab_files.upload()          # alternativa: montar Drive y ajustar DATA_PATH
    faltan = [f for f in DATA_FILES if not os.path.exists(f)]
if faltan:
    raise FileNotFoundError(f"Faltan archivos de datos: {faltan}")
DATA_PATH = DATA_FILES[0]
print("Datos listos:", DATA_FILES)

## 1. Carga de la base unificada y conciliación
Base: `261030_CHQ_unificado_LC_CA_PF_limpia.xlsx`, hoja **Base LCCA** (una fila por persona encuestada).
Olas: agosto 2024, agosto 2025 y tercer trimestre 2026 (la ola fuera de ciclo de febrero 2025 no está en la base).
Todas las variables del modelo se construyen según `libro_de_codigos.csv`.

In [ ]:
# Carga
import json, re, warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
warnings.filterwarnings("ignore")

RAW = pd.read_excel(DATA_PATH, sheet_name="Base LCCA")
COLS = [c for c in RAW.columns if isinstance(c, str)]

def col(prefijo):
    """Primera columna cuyo encabezado empieza con `prefijo` (los encabezados son las preguntas completas)."""
    m = [c for c in COLS if c.startswith(prefijo)]
    if not m:
        raise KeyError(prefijo)
    return m[0]

ACT = {"Población Flotante": "flotante", "Local comercial": "locales", "Comercio autónomo regularizado": "autonomo"}
ACTORES = ["flotante", "locales", "autonomo"]
NOMBRE = {"flotante": "Población flotante", "locales": "Locales comerciales", "autonomo": "Comercio autónomo"}
ANIOS = [2024, 2025, 2026]

B = RAW.copy()
B["anio"] = B["Año"].astype(int)
B["tipo_actor"] = B["Tipo"].map(ACT)
assert B["tipo_actor"].notna().all() and B["anio"].isin(ANIOS).all()

conc = B.groupby(["anio", "tipo_actor"]).size().unstack()[ACTORES]
print("Filas brutas:", len(RAW))
print(conc)

## 2. Variables del libro de códigos
Codificación binaria 1/0. `Ns/Nc` y respuestas vacías se tratan como valores perdidos (NaN).

In [ ]:
# Construcción de variables según libro_de_codigos.csv
def si_no(s):
    return s.map({"Si": 1, "No": 0}).astype(float)

data = pd.DataFrame({
    "anio": B["anio"],
    "tipo_actor": B["tipo_actor"],
    "inseguro": B[col("PSC.1 En")].map({"Inseguro": 1, "Muy inseguro": 1, "Seguro": 0, "Muy seguro": 0}).astype(float),
    "victima": si_no(B[col("VI.1")]),
    "vio_ffaa": si_no(B[col("SSM.8 ")]),
    "vio_policia": si_no(B[col("SSM.7 ")]),
    "vio_cacmq": si_no(B[col("SSM.3 ")]),
    "cc_peleas": si_no(B[col("CC.1 ")]),
    "cc_situacion_calle": si_no(B[col("CC.2 ")]),
    "cc_alcohol": si_no(B[col("CC.3 ")]),
    "cc_drogas": si_no(B[col("CC.4 ")]),
    "cc_danio_propiedad": si_no(B[col("CC.5 ")]),
    "cc_fauna_urbana": si_no(B[col("CC.6 ")]),
    "cc_basura": si_no(B[col("CC.7 ")]),
    "cc_ruido": si_no(B[col("CC.8 ")]),
    "cc_aglomeracion": si_no(B[col("CC.9 ")]),
    "ferias": si_no(B[col("CC.13 ")]),
    "hombre": B[col("DD. 1")].map({"Hombre": 1, "Mujer": 0}).astype(float),
})
# id anonimizado: CHQ{aa}_{correlativo por ola}
data.insert(0, "id", [f"CHQ{str(a)[2:]}_{i:04d}" for a, i in zip(data["anio"], data.groupby("anio").cumcount() + 1)])
VARS = list(data.columns)
data.to_csv(f"{OUT}/CHQ_percepcion_2024_2026_replicacion.csv", index=False)
print(data.shape)
print("Valores perdidos por variable:"); print(data.isna().sum()[data.isna().sum() > 0])

## 3. Indicadores descriptivos del HTML
Convenciones: porcentajes sobre respuestas válidas (se excluyen Ns/Nc y vacíos) salvo que se indique otra base.
Percepción, victimización y presencia institucional: % sobre todas las personas encuestadas (no tienen perdidos).

In [ ]:
# Utilidades
def pct(num, den):
    den = float(den)
    return float(num) / den * 100 if den else None

def p_val(s, positivos, validos=None):
    """% de `positivos` sobre respuestas válidas (`validos`; por defecto, no vacías)."""
    s = s.astype(str).str.strip()
    base = s.isin(validos) if validos is not None else (s != "nan")
    return pct(s[base].isin(positivos).sum(), base.sum())

SEG4 = ["Muy seguro", "Seguro", "Inseguro", "Muy inseguro"]
INS = ["Inseguro", "Muy inseguro"]
HAB = ["Ha disminuido", "Ha aumentado", "Permanece igual"]
CAL = ["Muy bueno", "Bueno", "Malo", "Muy malo"]
BUENO = ["Muy bueno", "Bueno"]

def sub(actor, anio):
    return B[(B["tipo_actor"] == actor) & (B["anio"] == anio)]

A = {"n": {}, "serie": {}, "actor": {}}
for y in ANIOS:
    A["n"][str(y)] = int((B["anio"] == y).sum())
    for a in ACTORES:
        A["n"][f"{a}_{y}"] = int(conc.loc[y, a])
A["n"]["total"] = int(len(B))

In [ ]:
# Serie general (promedio ponderado de los tres actores = media de la muestra)
for y in ANIOS:
    d = data[data["anio"] == y]
    A["serie"][str(y)] = {"inseguro": d["inseguro"].mean() * 100, "victima": d["victima"].mean() * 100}
A["serie"]["var_inseguro_24_26"] = A["serie"]["2026"]["inseguro"] - A["serie"]["2024"]["inseguro"]
A["serie"]["var_victima_25_26"] = A["serie"]["2026"]["victima"] - A["serie"]["2025"]["victima"]
print(pd.DataFrame(A["serie"]).round(1))

In [ ]:
# Percepción de inseguridad por actor
RAZONES = [("Delincuencia (robos", "Delincuencia"), ("Falta de UPC y policías", "Falta de UPC y policías"),
           ("Venta de drogas", "Venta de drogas"), ("Consumo de drogas", "Consumo de drogas"),
           ("Ventas ambulantes", "Ventas ambulantes"), ("Libadores en el espacio público", "Libadores en el espacio público"),
           ("Personas en situación de calle o habitantes de calle", "Personas en situación de calle"),
           ("Personas extranjeras", "Personas extranjeras"), ("Personas trabajadoras sexuales", "Personas trabajadoras sexuales"),
           ("Falta de alumbrado público", "Falta de alumbrado público"), ("Presencia de pandillas", "Presencia de pandillas o grupos GDO"),
           ("Otra", "Otra")]
PIN = [("PIN.9", "Caminando solo/a en la noche por este sector"), ("PIN.8", "Caminando solo/a en el día por este sector"),
       ("PIN.5", "Transitando por sitios con basura"), ("PIN.3", "En presencia de personas en situación de calle"),
       ("PIN.4", "En presencia de vendedores ambulantes"), ("PIN.7", "Utilizando cajeros automáticos en la vía pública"),
       ("PIN.6", "En espectáculos públicos o actividades recreativas"), ("PIN.2", "En calles y plazas del sector")]
HORARIOS = [("Madrugada (00:00", "Madrugada (00:00-05:59)"), ("Mañana  (06:00", "Mañana (06:00-11:59)"),
            ("Tarde (12:00", "Tarde (12:00-17:59)"), ("Noche  (18:00", "Noche (18:00-23:59)"), ("Todos los horarios", "Todos los horarios")]

def col_last(prefijo):
    m = [c for c in COLS if c.startswith(prefijo)]
    return m[-1]   # las columnas dummy de horario están después de PSC.3 y de 'horario 2/3'

for a in ACTORES:
    R = A["actor"].setdefault(a, {"nombre": NOMBRE[a]})
    R["inseguro"] = {str(y): data[(data.anio == y) & (data.tipo_actor == a)]["inseguro"].mean() * 100 for y in ANIOS}
    R["victima"] = {str(y): data[(data.anio == y) & (data.tipo_actor == a)]["victima"].mean() * 100 for y in ANIOS}
    # Razones de inseguridad (solo a quienes se sienten inseguros). 2025-2026: ranking hasta 3 (1 = principal)
    R["razones"] = {}
    for y in (2025, 2026):
        x = sub(a, y); xi = x[x[col("PSC.1 En")].isin(INS)]
        filas = [{"label": lab, "top3": pct(xi[col(p)].isin([1, 2, 3]).sum(), len(xi)),
                  "principal": pct((xi[col(p)] == 1).sum(), len(xi))} for p, lab in RAZONES]
        filas.sort(key=lambda r: (-r["top3"], -r["principal"]))
        R["razones"][str(y)] = {"n": int(len(xi)), "top": filas[:5]}
    # 2024: mención única
    x = sub(a, 2024); xi = x[x[col("PSC.1 En")].isin(INS)]
    mot = xi["Motivos de inseguridad"].astype(str).str.strip()
    R["razon_2024"] = {"n": int(len(xi)), "delincuencia": pct(mot.str.startswith("Delincuencia").sum(), len(xi))}
    # Evolución percibida (PSC.4): % que cree que la inseguridad empeoró / mejoró
    R["evolucion"] = {str(y): {"empeoro": p_val(sub(a, y)[col("PSC.4")], ["Empeoró"], ["Empeoró", "Mejoró", "Está igual"]),
                               "mejoro": p_val(sub(a, y)[col("PSC.4")], ["Mejoró"], ["Empeoró", "Mejoró", "Está igual"])} for y in ANIOS}
    if a == "flotante":
        # Percepción en situaciones específicas (PIN): solo 2025-2026 (en 2024 se preguntó solo a una submuestra)
        R["pin"] = [{"label": lab, **{str(y): p_val(sub(a, y)[col(p)], INS, SEG4) for y in (2025, 2026)}} for p, lab in PIN]
        R["pin"].sort(key=lambda r: -r["2026"])
    else:
        # Inseguridad dentro del local / en el puesto de trabajo (PSC.2)
        R["lugar_trabajo"] = {str(y): p_val(sub(a, y)[col("PSC. 2")], INS, SEG4) for y in ANIOS}
        # Horario de mayor riesgo (respuesta múltiple; % de personas encuestadas del actor)
        x = sub(a, 2026)
        hs = [{"label": lab, "pct": pct((x[col_last(p)] == 1).sum(), x[col_last(p)].notna().sum())} for p, lab in HORARIOS]
        R["horario_2026"] = sorted(hs, key=lambda r: -r["pct"])

In [ ]:
# Victimización: tipos de delito y denuncia (base: víctimas)
def norm_delito(s):
    s = re.sub(r"\s+", " ", str(s)).strip()
    if s == "nan" or not s:
        return None
    s = s[0].upper() + s[1:].lower()
    s = s.replace("Robo accesorios de vehículo", "Robo de accesorios de vehículo")
    return s

for a in ACTORES:
    R = A["actor"][a]; R["denuncia"] = {}; R["delitos"] = {}
    for y in ANIOS:
        x = sub(a, y); v = x[x[col("VI.1")] == "Si"]
        R["denuncia"][str(y)] = pct((v[["Denuncia 1", "Denuncia 2", "Denuncia 3"]] == "Si").any(axis=1).sum(), len(v))
        if y == 2026:
            tipos = v[["Tipo delito 1", "Tipo de delito 2", "Tipo de delito 3"]].map(norm_delito)
            etiquetas = sorted({t for t in tipos.values.ravel() if isinstance(t, str)})
            filas = [{"label": t, "pct": pct((tipos == t).any(axis=1).sum(), len(v))} for t in etiquetas]
            filas.sort(key=lambda r: (-r["pct"], r["label"]))
            R["delitos"] = {"n": int(len(v)), "top": filas[:5]}

In [ ]:
# Cambios de hábito (módulo desde 2025): % que dice que el hábito 'ha disminuido'
HABITOS = {"flotante": [("FCH. 1", "Transita solo/a por el sector"), ("FCH. 2", "Porta dinero en efectivo"),
                        ("FCH. 3", "Frecuenta restaurantes/cafeterías"), ("FCH. 4", "Usa cajeros automáticos")],
           "locales": [("CH.1", "Atiende la venta al público"), ("CH.2", "Transita solo/a en el sector"), ("CH.3", "Implementa medidas de seguridad")],
           "autonomo": [("CH.1", "Atiende la venta al público"), ("CH.2", "Transita solo/a al lugar de trabajo"), ("CH.3", "Implementa medidas de seguridad")]}
for a in ACTORES:
    A["actor"][a]["habitos"] = [{"label": lab, **{str(y): p_val(sub(a, y)[col(p)], ["Ha disminuido"], HAB) for y in (2025, 2026)},
                                 "aumento_2026": p_val(sub(a, 2026)[col(p)], ["Ha aumentado"], HAB)} for p, lab in HABITOS[a]]

In [ ]:
# Convivencia ciudadana: % que presenció cada fenómeno (Si / Si+No)
CCV = [("CC.1 ", "Peleas o riñas en el espacio público"), ("CC.2 ", "Personas en situación de calle (mendicidad)"),
       ("CC.3 ", "Consumo de alcohol en el espacio público"), ("CC.4 ", "Consumo de drogas en el espacio público"),
       ("CC.5 ", "Daño a la propiedad pública y privada"), ("CC.6 ", "Fauna urbana (animales sin hogar)"),
       ("CC.7 ", "Basura en los espacios públicos"), ("CC.8 ", "Ruidos molestos"), ("CC.9 ", "Aglomeración de personas")]
for a in ACTORES:
    R = A["actor"][a]
    filas = [{"label": lab, **{str(y): p_val(sub(a, y)[col(p)], ["Si"], ["Si", "No"]) for y in ANIOS}} for p, lab in CCV]
    R["convivencia"] = sorted(filas, key=lambda r: -r["2026"])
    R["ferias"] = {**{str(y): p_val(sub(a, y)[col("CC.13 ")], ["Si"], ["Si", "No"]) for y in ANIOS},
                   "calif_2026": p_val(sub(a, 2026)[col("CC.13.1")], BUENO, CAL)}
    if a != "flotante":   # control del comercio informal (desde 2025)
        x26 = sub(a, 2026); si = x26[col("CC.11 ")] == "Si"
        R["control_informal"] = {str(y): p_val(sub(a, y)[col("CC.11 ")], ["Si"], ["Si", "No"]) for y in (2025, 2026)}
        R["control_informal"]["adecuada_2026"] = p_val(x26.loc[si, col("CC. 11.1")], ["Adecuada"], ["Adecuada", "Parcial"])
        R["control_informal"]["parcial_2026"] = p_val(x26.loc[si, col("CC. 11.1")], ["Parcial"], ["Adecuada", "Parcial"])

In [ ]:
# Satisfacción de servicios institucionales (SSM) y brigadas de seguridad (EOS)
INST = [("SSM.1", "Agencia Metropolitana de Control (AMC)"), ("SSM.2", "Patronato San José (UPMSJ)"),
        ("SSM.3", "Cuerpo de Agentes de Control Metropolitano de Quito (CACMQ)"), ("SSM.4", "Cuerpo de Bomberos de Quito (CBQ)"),
        ("SSM.5", "Agencia Metropolitana de Tránsito (AMT)"), ("SSM.6", "Empresa Metropolitana de Aseo (EMASEO)"),
        ("SSM.7", "Policía Nacional (PN)"), ("SSM.8", "Fuerzas Armadas del Ecuador (FFAA)"),
        ("SSM.9", "Secretaría General de Seguridad Ciudadana y Gestión de Riesgos (SGSCGR)"),
        ("SSM.10", "Administración Zonal Manuela Sáenz"), ("SSM.11", "Agencia Metropolitana de Coordinación de Comercio"),
        ("SSM.12", "Quito Turismo")]
for a in ACTORES:
    filas = []
    for k, lab in INST:
        x26, x24 = sub(a, 2026), sub(a, 2024)
        pres26 = p_val(x26[col(k + " ")], ["Si"], ["Si", "No"])
        if pres26 is None:
            continue   # institución no preguntada a este actor
        filas.append({"label": lab, "pres_2026": pres26, "pres_2024": p_val(x24[col(k + " ")], ["Si"], ["Si", "No"]),
                      "beneficio": p_val(x26[col(k + ".1")], ["Si"], ["Si", "No"]),
                      "calidad": p_val(x26[col(k + ".2")], BUENO, CAL),
                      "contribuyo": p_val(x26[col(k + ".3")], ["Si"], ["Si", "No"])})
    A["actor"][a]["instituciones"] = sorted(filas, key=lambda r: -r["pres_2026"])
    A["actor"][a]["brigadas"] = {str(y): {"presencio": p_val(sub(a, y)[col("EOS.1")], ["Si"], ["Si", "No"]),
                                          "calificacion": p_val(sub(a, y)[col("En una escala del 1 al 4")], BUENO, CAL),
                                          "ayudaron": p_val(sub(a, y)[col("EOS.2 ")], ["Si"], ["Si", "No"]),
                                          "confia": p_val(sub(a, y)[col("EOS.4 ")], ["Si"], ["Si", "No"])} for y in (2025, 2026)}

print(json.dumps({a: {k: A["actor"][a][k] for k in ("inseguro", "victima", "denuncia")} for a in ACTORES}, indent=1, default=lambda v: round(v, 1)))

## 4. Modelo logístico agrupado (2024 + 2026) — efectos marginales promedio
Variable dependiente `inseguro`. Solo olas 2024 y 2026; se eliminan observaciones con valores perdidos en alguna covariable.
Categoría de referencia del tipo de actor: comercio autónomo. Errores estándar robustos HC1.

In [ ]:
# Muestra analítica
CC = ["cc_peleas", "cc_situacion_calle", "cc_alcohol", "cc_drogas", "cc_danio_propiedad",
      "cc_fauna_urbana", "cc_basura", "cc_ruido", "cc_aglomeracion"]
INSTV = ["vio_cacmq", "vio_policia", "vio_ffaa"]
df = data[data["anio"].isin([2024, 2026])].dropna().copy()
df["anio2026"] = (df["anio"] == 2026).astype(int)
df["actor_flotante"] = (df["tipo_actor"] == "flotante").astype(int)
df["actor_locales"] = (df["tipo_actor"] == "locales").astype(int)
COV = ["anio2026", "hombre", "victima", "ferias"] + CC + INSTV + ["actor_flotante", "actor_locales"]
ETIQ = {"anio2026": "Año 2026 (vs. 2024, a igual composición)", "hombre": "Ser hombre (vs. mujer)",
        "victima": "Haber sido víctima de un delito", "ferias": "Haber presenciado ferias gastronómicas/culturales",
        "cc_peleas": "Peleas o riñas en el espacio público", "cc_situacion_calle": "Personas en situación de calle",
        "cc_alcohol": "Consumo de alcohol en el espacio público", "cc_drogas": "Consumo de drogas en el espacio público",
        "cc_danio_propiedad": "Daño a la propiedad pública/privada", "cc_fauna_urbana": "Fauna urbana (animales sin hogar)",
        "cc_basura": "Basura en los espacios públicos", "cc_ruido": "Ruidos molestos", "cc_aglomeracion": "Aglomeración de personas",
        "actor_flotante": "Ser población flotante (vs. comercio autónomo)", "actor_locales": "Ser local comercial (vs. comercio autónomo)",
        "vio_cacmq": "Vio trabajar al CACMQ", "vio_policia": "Vio trabajar a la Policía Nacional", "vio_ffaa": "Vio trabajar a las Fuerzas Armadas"}
excluidas = data[data["anio"].isin([2024, 2026])].groupby("anio").size() - df.groupby("anio").size()
M = {"n": int(len(df)), "n_2024": int((df.anio == 2024).sum()), "n_2026": int((df.anio == 2026).sum()),
     "n_panel_2024": int((data.anio == 2024).sum()), "n_panel_2026": int((data.anio == 2026).sum()),
     "excluidas": {str(k): int(v) for k, v in excluidas.items()}}
for y in (2024, 2026):
    M[f"inseguro_{y}"] = df.loc[df.anio == y, "inseguro"].mean() * 100
    M[f"victima_{y}"] = df.loc[df.anio == y, "victima"].mean() * 100
print(M)

In [ ]:
# Logit agrupado y AME
def ajustar(d, cov):
    X = sm.add_constant(d[cov].astype(float), has_constant="add")
    return sm.Logit(d["inseguro"].astype(float), X).fit(disp=0, cov_type="HC1")

m = ajustar(df, COV)
ame = m.get_margeff(at="overall").summary_frame().rename(columns={"dy/dx": "AME", "Std. Err.": "EE", "Pr(>|z|)": "p"})
ame = ame.reindex(ame["AME"].abs().sort_values(ascending=False).index)
M["pseudo_r2"] = float(m.prsquared)
M["ame"] = [{"var": v, "label": ETIQ[v], "ame": float(ame.loc[v, "AME"] * 100), "ee": float(ame.loc[v, "EE"] * 100),
             "p": float(ame.loc[v, "p"])} for v in ame.index]
print(f"N = {int(m.nobs)} | pseudo-R2 = {m.prsquared:.3f}")
print(pd.DataFrame(M["ame"]).set_index("label")[["ame", "p"]].round(3))

## 5. Descomposición Oaxaca-Blinder/Fairlie de la brecha 2024-2026
Dos logit separados (2024 y 2026) con las mismas covariables, sin indicador de año. Se reportan ambas referencias de coeficientes.
Detalle por variable: aproximación lineal de Yun (2004); valores positivos = contribuyen a la mejora (menos inseguridad).

In [ ]:
# Descomposición
DCOV = [c for c in COV if c != "anio2026"]

def descomponer(d24, d26):
    X24 = sm.add_constant(d24[DCOV].astype(float), has_constant="add")
    X26 = sm.add_constant(d26[DCOV].astype(float), has_constant="add")
    m24 = sm.Logit(d24["inseguro"].astype(float), X24).fit(disp=0, cov_type="HC1")
    m26 = sm.Logit(d26["inseguro"].astype(float), X26).fit(disp=0, cov_type="HC1")
    P = lambda mod, X: mod.predict(X).mean()
    p24, p26 = P(m24, X24), P(m26, X26)
    r = {"p24": p24, "p26": p26, "brecha": p24 - p26,
         "expl_24": p24 - P(m24, X26), "noexpl_24": P(m24, X26) - p26,
         "expl_26": P(m26, X24) - p26, "noexpl_26": p24 - P(m26, X24), "m24": m24, "m26": m26}
    xbar = pd.concat([X24, X26]).mean()
    for ref, mod in (("24", m24), ("26", m26)):
        pb = 1 / (1 + np.exp(-(xbar * mod.params).sum()))
        r[f"detalle_{ref}"] = {k: -mod.params[k] * (X26[k].mean() - X24[k].mean()) * pb * (1 - pb) * 100 for k in DCOV}
    return r

d24, d26 = df[df["anio"] == 2024], df[df["anio"] == 2026]
D = descomponer(d24, d26)
g = D["brecha"]
det = pd.DataFrame({"ref24": D["detalle_24"], "ref26": D["detalle_26"]})
det["orden24"] = det["ref24"].rank(ascending=False).astype(int)
det["orden26"] = det["ref26"].rank(ascending=False).astype(int)
medias = df.groupby("anio")[DCOV].mean() * 100
M["desc"] = {"p24": D["p24"] * 100, "p26": D["p26"] * 100, "brecha": g * 100,
             "expl_24": D["expl_24"] * 100, "noexpl_24": D["noexpl_24"] * 100,
             "expl_26": D["expl_26"] * 100, "noexpl_26": D["noexpl_26"] * 100,
             "pct_expl_24": D["expl_24"] / g * 100, "pct_expl_26": D["expl_26"] / g * 100,
             "pct_noexpl_24": D["noexpl_24"] / g * 100, "pct_noexpl_26": D["noexpl_26"] / g * 100,
             "detalle": [{"var": k, "label": ETIQ[k], "ref24": float(det.loc[k, "ref24"]), "ref26": float(det.loc[k, "ref26"]),
                          "orden24": int(det.loc[k, "orden24"]), "orden26": int(det.loc[k, "orden26"]),
                          "media_2024": float(medias.loc[2024, k]), "media_2026": float(medias.loc[2026, k]),
                          "beta24": float(D["m24"].params[k]), "p24": float(D["m24"].pvalues[k]),
                          "beta26": float(D["m26"].params[k]), "p26": float(D["m26"].pvalues[k])} for k in DCOV]}
print(f"P(inseguro) 2024 = {D['p24']*100:.1f}% | 2026 = {D['p26']*100:.1f}% | brecha = {g*100:.1f} p.p.")
print(pd.DataFrame(M["desc"]["detalle"]).set_index("label")[["ref24", "orden24", "ref26", "orden26", "media_2024", "media_2026", "beta24", "beta26"]].sort_values("ref24", ascending=False).round(2))
print({k: round(v, 2) for k, v in M["desc"].items() if k != "detalle"})

## 6. Intervalos de confianza bootstrap de la descomposición
Bootstrap no paramétrico que remuestrea personas dentro de cada ola (B = 300, semilla fija 20260924).

In [ ]:
# Bootstrap
rng = np.random.default_rng(20260924)
NB, boot = 300, []
for b in range(NB):
    s24 = d24.sample(len(d24), replace=True, random_state=rng.integers(1e9))
    s26 = d26.sample(len(d26), replace=True, random_state=rng.integers(1e9))
    try:
        r = descomponer(s24, s26)
        fila = {"brecha": r["brecha"] * 100, "pct_expl_24": r["expl_24"] / r["brecha"] * 100, "pct_expl_26": r["expl_26"] / r["brecha"] * 100}
        for k in DCOV:
            fila[f"{k}_24"] = r["detalle_24"][k]; fila[f"{k}_26"] = r["detalle_26"][k]
        boot.append(fila)
    except Exception:
        pass
boot = pd.DataFrame(boot)
q = boot.quantile([0.025, 0.975])
M["boot"] = {"replicas": int(len(boot)), "B": NB,
             **{c: {"lo": float(q.loc[0.025, c]), "hi": float(q.loc[0.975, c])} for c in boot.columns}}
print(f"Réplicas exitosas: {len(boot)}/{NB}")
print(q[["brecha", "pct_expl_24", "pct_expl_26"]].T.round(2))
print(q[[c for c in boot.columns if c.split("_")[0] in ("cc", "vio")]].T.round(2))
A["modelo"] = M

## 7. Diagnóstico por actor (selector de la sección 7 del HTML) y salida
El dict `A` contiene todas las cifras que dibuja el HTML y las que cita el informe técnico.

In [ ]:
# Diagnóstico por actor
for a in ACTORES:
    R = A["actor"][a]
    raz = R["razones"]["2026"]["top"][0]
    inst = R["instituciones"][0]
    R["diag"] = {"kpis": [{"lab": "Percepción insegura", "val": R["inseguro"]["2026"]},
                          {"lab": "Victimización", "val": R["victima"]["2026"]},
                          {"lab": "Tasa de denuncia", "val": R["denuncia"]["2026"]}],
                 "bars": [{"label": f["label"], "pct": f["2026"]} for f in R["convivencia"][:3]],
                 "razon": raz, "inst": {"label": inst["label"], "pct": inst["pres_2026"]},
                 "evol": R["evolucion"]["2026"], "confia": R["brigadas"]["2026"]["confia"]}
A["meta"] = {"base": "261030_CHQ_unificado_LC_CA_PF_limpia.xlsx", "hoja": "Base LCCA", "filas_brutas": int(len(RAW))}
print("Listo: A con", len(json.dumps(A, default=float)), "caracteres")

## 8. Zona emergente: entorno de La Basílica (2026)
Base aparte (`Unificado_Basilica_2026.xlsx`, hojas *PF Basílica* y *LC Basílica*). **No se unifica** con la base del Centro Histórico:
es un espacio distinto, medido por primera vez en 2026. Las respuestas traen prefijo numérico ("1. Si"), que se elimina antes de codificar.
`inseguro` y `victima` siguen el libro de códigos; el contraste es con el núcleo del Centro Histórico en 2026.

In [ ]:
# La Basílica
DATA_PATH_BAS = DATA_FILES[1]

def limpiar(s):
    return s.astype(str).str.replace(r"^\s*\d+\.\s*", "", regex=True).str.strip().str.lower() \
            .str.replace("sí", "si").replace("nan", np.nan)

def pb(s, pos, val):
    s = limpiar(s); base = s.isin(val)
    return pct(s[base].isin(pos).sum(), base.sum())

A["basilica"] = {}
for hoja, a in (("PF Basílica", "flotante"), ("LC Basílica", "locales")):
    X = pd.read_excel(DATA_PATH_BAS, sheet_name=hoja)
    XC = [c for c in X.columns if isinstance(c, str)]
    xc = lambda pref: [c for c in XC if c.startswith(pref)][0]
    ins = limpiar(X[[c for c in XC if re.match(r"PSC\.? ?1\.? En", c)][0]])
    vic = limpiar(X[xc("VI.1")])
    den_cols = [c for c in XC if "Denunció" in c]
    victimas = X[vic == "si"]
    Bz = {"n": int(len(X)),
          "inseguro": pct(ins.isin(["inseguro", "muy inseguro"]).sum(), ins.isin(["inseguro", "muy inseguro", "seguro", "muy seguro"]).sum()),
          "victima": pct((vic == "si").sum(), vic.isin(["si", "no"]).sum()),
          "n_victimas": int(len(victimas)),
          "denuncia": pct((victimas[den_cols].apply(limpiar) == "si").any(axis=1).sum(), len(victimas)),
          "pmo_presencio": pb(X[xc("EOS.1")], ["si"], ["si", "no"]),
          "calle": pb(X[xc("CC.2 ")], ["si"], ["si", "no"]),
          "nucleo_inseguro": A["actor"][a]["inseguro"]["2026"], "nucleo_victima": A["actor"][a]["victima"]["2026"]}
    conf = [c for c in XC if c.startswith("EOS.") and "confía" in c]
    Bz["pmo_confia"] = pb(X[conf[0]], ["si"], ["si", "no"])
    if a == "locales":
        Bz["control_informal"] = pb(X[xc("CC.11")], ["si"], ["si", "no"])
    A["basilica"][a] = Bz
print(json.dumps(A["basilica"], indent=1, default=lambda v: round(v, 1)))

In [ ]:
# Tabla maestra: cada cifra de APP_DATA (de aquí salen texto, tablas, barras y KPI del HTML)
import json, pandas as pd

def _aplanar(o, ruta=""):
    if isinstance(o, dict):
        for k, v in o.items(): yield from _aplanar(v, f"{ruta}.{k}" if ruta else k)
    elif isinstance(o, list):
        for i, v in enumerate(o): yield from _aplanar(v, f"{ruta}.{i}")
    else:
        yield ruta, o

cifras = pd.DataFrame(list(_aplanar(A)), columns=["ruta", "valor"])
cifras.to_csv(f"{OUT}/cifras_html.csv", index=False)
json.dump(A, open(f"{OUT}/app_data.json", "w"), ensure_ascii=False, indent=1, default=str)
print(len(cifras), "cifras →", f"{OUT}/cifras_html.csv y app_data.json")
cifras.head(40)

In [ ]:
# Verificación contra el HTML publicado (opcional): sube el .html a Colab para correrla
import re, json, os, math
HTML = "261030_CHQ_percepcion_convivencia_servicios_v4.html"
if not os.path.exists(HTML) and IN_COLAB:
    print("Sube", HTML, "para comparar (o ignora esta celda)"); _colab_files.upload()
if os.path.exists(HTML):
    src = open(HTML, encoding="utf-8").read()
    m = re.search(r'<script id="app-data" type="application/json">(.*?)</script>', src, re.S)
    if not m: raise SystemExit("El HTML no incrusta APP_DATA como <script id='app-data'>")
    H = json.loads(m.group(1))
    def _cmp(a, b, ruta, dif):
        if isinstance(a, dict) and isinstance(b, dict):
            for k in a:
                if k in b: _cmp(a[k], b[k], f"{ruta}.{k}" if ruta else k, dif)
        elif isinstance(a, list) and isinstance(b, list):
            for i, (x, y) in enumerate(zip(a, b)): _cmp(x, y, f"{ruta}.{i}", dif)
        elif isinstance(a, (int, float)) and isinstance(b, (int, float)) and not isinstance(a, bool):
            if not math.isclose(a, b, rel_tol=1e-9, abs_tol=1e-9): dif.append((ruta, a, b))
        elif a != b: dif.append((ruta, a, b))
    dif = []; _cmp(A, H, "", dif)
    print("Coinciden todas las cifras comunes" if not dif else f"{len(dif)} diferencias:")
    for d in dif[:50]: print("  ", d)
else:
    print("HTML no encontrado; verificación omitida")

## 9. Cuadros del informe técnico
Reproduce los Cuadros 1, 2 y 3 del informe `261030_informe_modelo_percepcion_CHQ_v4.docx` desde `A` (mismas cifras que el texto).

In [ ]:
# Cuadros 1-3 del informe
fmt = lambda x, d=1: f"{x:.{d}f}".replace(".", ",")
cuadro1 = pd.DataFrame([[A["actor"][a]["nombre"], " → ".join(fmt(A["actor"][a]["inseguro"][y]) + "%" for y in ("2024", "2025", "2026")),
                         " → ".join(fmt(A["actor"][a]["victima"][y]) + "%" for y in ("2024", "2025", "2026"))] for a in ACTORES],
                       columns=["Actor", "Percepción insegura 2024 → 2025 → 2026", "Victimización 2024 → 2025 → 2026"])
est = lambda p: "**" if p < 0.01 else ("*" if p < 0.05 else "")
cuadro2 = pd.DataFrame([[r["label"], f"{'+' if r['ame'] >= 0 else ''}{fmt(r['ame'])} p.p.{est(r['p'])}", fmt(r["p"], 3)] for r in A["modelo"]["ame"]],
                       columns=["Variable", "Efecto marginal", "Valor p"])
Dd = A["modelo"]["desc"]
cuadro3 = pd.DataFrame({"Referencia: coeficientes 2024": [f"{fmt(Dd['expl_24'])} p.p. ({fmt(Dd['pct_expl_24'])}%)", f"{fmt(Dd['noexpl_24'])} p.p. ({fmt(Dd['pct_noexpl_24'])}%)"],
                        "Referencia: coeficientes 2026": [f"{fmt(Dd['expl_26'])} p.p. ({fmt(Dd['pct_expl_26'])}%)", f"{fmt(Dd['noexpl_26'])} p.p. ({fmt(Dd['pct_noexpl_26'])}%)"]},
                       index=["Explicado por composición", "No explicado (cambio de relación)"])
print(cuadro1.to_string(index=False)); print(); print(cuadro2.to_string(index=False)); print(); print(cuadro3)